Ячейка 1 (Импорты):

In [1]:
import sys
sys.path.append('..')
import pandas as pd
from IPython.display import display
from src.cleaning import *

Ячейка 2 (Текст + Загрузка):

In [2]:
df = pd.read_parquet('../data/interim/full_data_optimized_full.parquet').head(500_000)
initial_len = len(df)
print(f"Строк до очистки: {initial_len}")

Строк до очистки: 500000


Ячейка 3 (Нормализация имен + Текст):

In [3]:
df = normalize_columns(df)
display(df.head(2))

INFO: Нормализация имен столбцов...


,callsign,number,aircraft_uid,typecode,origin,destination,firstseen,lastseen,day,latitude_1,longitude_1,altitude_1,latitude_2,longitude_2,altitude_2
0,HVN19,NaN,947f9b63-4fee-499b-870b-57cd8a32d468,NaN,YMML,LFPG,2018-12-31 00:43:16+00:00,2019-01-01 04:56:29+00:00,2019-01-01 00:00:00+00:00,-37.659485,144.804428,304.799988,48.995316,2.610802,-53.34000
1,CCA839,NaN,7f6d394a-6882-4113-b7a2-b56b5c134367,NaN,YMML,LEBL,2018-12-31 00:53:08+00:00,2019-01-01 06:01:42+00:00,2019-01-01 00:00:00+00:00,-37.692123,144.841995,304.799988,41.036125,2.063557,1112.52002


Ячейка 4 (Типы + Текст):

In [4]:
df = cast_types(df)
display(df.dtypes.to_frame('Типы после оптимизации'))

INFO: Приведение типов данных...


,Типы после оптимизации
callsign,category
number,str
aircraft_uid,string
typecode,category
origin,category
destination,category
firstseen,"datetime64[us, UTC]"
lastseen,"datetime64[us, UTC]"
day,"datetime64[us, UTC]"
latitude_1,float32


Ячейка 5 (Пропуски + Текст):

In [5]:
df = handle_hidden_nulls(df)
print(f"Строк до обработки пропусков: {len(df)}")
df = analyze_missing(df)
print(f"Строк после обработки пропусков: {len(df)}")

INFO: Поиск и замена скрытых пропусков...
INFO: Анализ структуры пропусков...
INFO: Удалено строк с пустыми origin/destination: 240981


Строк до обработки пропусков: 500000

Доля пропусков по столбцам (%):
number         99.9982
origin         30.6376
destination    25.7068
typecode       20.5106
callsign       17.7732
altitude_2      3.0460
latitude_2      0.0018
longitude_2     0.0018
dtype: float64
Строк после обработки пропусков: 259019


Ячейка 6 (Дубликаты + Текст):

In [6]:
df = drop_duplicates(df)

INFO: Удаление дубликатов...
INFO: Удалено полных дубликатов: 65
INFO: Удалено дубликатов по ключу: 38


Ячейка 7 (Категории и текст + Текст):

In [7]:
df = normalize_categories(df)
df = clean_text_columns(df)
display(df.head(2))

INFO: Нормализация категориальных столбцов...
INFO: Очистка текстовых столбцов...


,callsign,number,aircraft_uid,typecode,origin,destination,firstseen,lastseen,day,latitude_1,longitude_1,altitude_1,latitude_2,longitude_2,altitude_2
0,HVN19,NaN,947f9b63-4fee-499b-870b-57cd8a32d468,NaN,YMML,LFPG,2018-12-31 00:43:16+00:00,2019-01-01 04:56:29+00:00,2019-01-01 00:00:00+00:00,-37.659485,144.804428,304.799988,48.995316,2.610802,-53.34000
1,CCA839,NaN,7f6d394a-6882-4113-b7a2-b56b5c134367,NaN,YMML,LEBL,2018-12-31 00:53:08+00:00,2019-01-01 06:01:42+00:00,2019-01-01 00:00:00+00:00,-37.692123,144.841995,304.799988,41.036125,2.063557,1112.52002


Ячейка 8 (Выбросы и итог + Текст):

In [8]:
df = fix_outliers(df)
df = check_business_rules(df)

final_len = len(df)
print(f"Строк после очистки: {final_len}")
print(f"Потеряно строк: {initial_len - final_len} ({(1 - final_len/initial_len)*100:.2f}%)")

INFO: Обработка выбросов...
INFO: Найдено некорректных высот (правило предметной области): 33257
INFO: Найдено выбросов по правилу IQR: 8898 (оставляем их, так как они могут быть реальными)
INFO: Проверка бизнес-правил...
INFO: Найдено дат в будущем (firstseen): 0
INFO: Найдено некорректных широт (latitude_1): 0


Строк после очистки: 258916
Потеряно строк: 241084 (48.22%)


из терминал pipeline для сравнения:

INFO: Загрузка данных из data/interim/full_data_optimized_full.parquet...

INFO: Режим SAMPLE: взяты первые 500 000 строк.

INFO: Строк до очистки: 500000

INFO: Нормализация имен столбцов...

INFO: Приведение типов данных...

INFO: Поиск и замена скрытых пропусков...

INFO: Анализ структуры пропусков...

Доля пропусков по столбцам (%):

number         99.9982

origin         30.6376

destination    25.7068

typecode       20.5106

callsign       17.7732

altitude_2      3.0460

latitude_2      0.0018

longitude_2     0.0018

dtype: float64

INFO: Удалено строк с пустыми origin/destination: 240981

INFO: Удаление дубликатов...

INFO: Удалено полных дубликатов: 65

INFO: Удалено дубликатов по ключу: 38

INFO: Нормализация категориальных столбцов...

INFO: Очистка текстовых столбцов...

INFO: Обработка выбросов...

INFO: Найдено некорректных высот (правило предметной области): 33257

INFO: Найдено выбросов по правилу IQR: 8898 (оставляем их, так как они могут быть реальными)

INFO: Проверка бизнес-правил...

INFO: Найдено дат в будущем (firstseen): 0

INFO: Найдено некорректных широт (latitude_1): 0

INFO: Строк после очистки: 258916

INFO: Потеряно строк: 241084 (48.22%)

INFO: Сохранение очищенного набора в data/processed/clean_sample_test.parquet...

INFO: Готово!

Вывод:В ходе выполнения шага был реализован сквозной конвейер обработки данных в двух представлениях:

Исполняемый модуль (src/pipeline.py) с интерфейсом командной строки на основе библиотеки argparse. Скрипт поддерживает передачу путей к исходному файлу и результату, а также режим работы (--mode full для полного набора и --mode sample для отладочного среза). Обработка ошибок (отсутствие файла, неверный формат) реализована через проверки и логирование.

Интерактивный блокнот (notebooks/02_cleaning.ipynb), в котором каждый этап очистки вынесен в отдельную ячейку с текстовыми пояснениями. Результаты промежуточных шагов выводятся через display для визуального контроля.

Вся прикладная логика (функции очистки) вынесена в общий модуль src/cleaning.py. Блокнот и скрипт не дублируют код, а лишь вызывают одни и те же функции.

Результаты обязательных проверок:

Проверка согласованности: Блокнот и скрипт были запущены на одинаковом отладочном срезе (первые 500 000 строк файла full_data_optimized_full.parquet). Промежуточные результаты (в частности, количество найденных некорректных высот — 33 257, и итоговое число строк после очистки) полностью совпали. Это подтверждает, что логика не разошлась.

Проверка идемпотентности: Повторный запуск скрипта на тех же исходных данных выдает идентичный результат. Повторное применение функций очистки к уже очищенному набору не вызывает ошибок и не портит данные.

Проверка воспроизводимости: При удалении содержимого каталогов data/interim и data/processed и повторном запуске скрипта все файлы корректно восстанавливаются одной командой. Каталог data/raw при этом остается неизменным, что гарантирует возможность отследить происхождение любой записи.